<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎬 SoulX FlashHead - AI Talking Head Generator</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Audio-Driven Portrait Animation - Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0 0 20px 0;'>Free on Google Colab T4 GPU | Open Source</p>
</div>

---

<div align="center">

  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Colab-Free%20Tier-orange?style=for-the-badge&logo=googlecolab&logoColor=white" />

 <br>

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/▶%20Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20𝕏-0000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---


### About This Notebook

This notebook runs **SoulX FlashHead both Lite & Pro** — an audio-driven talking head generator with a full Gradio interface:

| Feature | Description |
|----|----|  
| **Lite & Pro Models** | Choose between fast Lite or high-quality Pro mode |
| **Audio-Driven Animation** | Animate portraits with any audio/speech input |
| **Wav2Vec2 Integration** | Facebook's wav2vec2 for audio feature extraction |
| **Gradio Web UI** | Easy-to-use interface with shareable public link |
| **Colab Optimized** | Runs on free T4 GPU with no restart needed |

### Before You Start
> Go to **Runtime -> Change runtime type -> T4 GPU**

In [ ]:
# @title Step 1: 🔍 GPU Check & PyTorch Verification
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
print()

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    try:
        x = torch.zeros(1, device="cuda")
        del x
        print("\n✅ PyTorch CUDA is working!")
    except Exception as e:
        print(f"\n⚠️ CUDA test failed: {e}")
else:
    print("\n❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# @title Step 2: 📥 Clone Repo & Install System Dependencies
import os

print("📦 Installing system dependencies...")
!apt-get install -y ffmpeg -q

print("\n📥 Cloning SoulX-FlashHead repository...")
if os.path.exists("/content/SoulX-FlashHead"):
    print("📂 Repository exists, pulling latest...")
    %cd /content/SoulX-FlashHead
    !git pull
else:
    !git clone https://github.com/Soul-AILab/SoulX-FlashHead.git
    %cd /content/SoulX-FlashHead

!ls
print("\n✅ Repository cloned & system deps installed!")

In [ ]:
# @title Step 3: ⚙️ Install Python Dependencies
# Strategy: Do NOT replace Colab's default torch. Install deps on top of it.

print("📦 Installing xformers...")
!pip install -q xformers

print("\n📦 Installing core dependencies (2-3 minutes)...")
!pip install -q \
    "opencv-python-headless>=4.12.0.88" \
    "diffusers>=0.34.0" \
    "transformers==4.57.3" \
    "tokenizers>=0.20.3" \
    "accelerate>=1.8.1" \
    tqdm imageio easydict ftfy \
    "imageio-ffmpeg" \
    scikit-image \
    loguru \
    "gradio>=5.0.0" \
    pyloudnorm \
    decord \
    librosa \
    "mediapipe>=0.10.13" \
    flask \
    huggingface_hub

print("\n📦 Installing xfuser...")
!pip install -q "xfuser>=0.4.3" || pip install -q "xfuser>=0.4.3" --no-deps
!pip install -q yunchang distvae sentencepiece beautifulsoup4 einops 2>/dev/null || true

print("\n✅ All dependencies installed!")
print("─" * 50)

import torch
print(f"🔧 torch: {torch.__version__}")
print(f"🔧 CUDA available: {torch.cuda.is_available()}")
try:
    import xformers; print(f"🔧 xformers: {xformers.__version__}")
except: print("⚠️ xformers: NOT FOUND")
try:
    import xfuser; print(f"🔧 xfuser: OK ({xfuser.__version__})")
except Exception as e: print(f"⚠️ xfuser: FAILED — {e}")
try:
    import diffusers; print(f"🔧 diffusers: {diffusers.__version__}")
except: print("⚠️ diffusers: NOT FOUND")

In [ ]:
# @title Step 4: 📥 Download Models
import os

CKPT_DIR    = "/content/models/SoulX-FlashHead-1_3B"
WAV2VEC_DIR = "/content/models/wav2vec2-base-960h"

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(WAV2VEC_DIR, exist_ok=True)

print("📥 Downloading SoulX-FlashHead-1_3B...")
!huggingface-cli download Soul-AILab/SoulX-FlashHead-1_3B \
    --local-dir {CKPT_DIR} 2>&1 | grep -E "Fetching|Downloading|100%|%\|"

print("\n📥 Downloading wav2vec2-base-960h...")
!huggingface-cli download facebook/wav2vec2-base-960h \
    --local-dir {WAV2VEC_DIR} 2>&1 | grep -E "Fetching|Downloading|100%|%\|"

# Quick verify
for name, path in [("FlashHead", CKPT_DIR), ("wav2vec2", WAV2VEC_DIR)]:
    if os.path.exists(path) and len(os.listdir(path)) > 0:
        print(f"✅ {name} downloaded")
    else:
        print(f"❌ {name} MISSING — check above for errors")

In [ ]:
# @title Step 5: 🔧 Patch All Files (Paths + Share + 16:9 Output)
import os, re
os.chdir("/content/SoulX-FlashHead")

# Reset ALL files to clean state
!git checkout -- gradio_app.py 2>/dev/null
!git checkout -- flash_head/utils/facecrop.py 2>/dev/null
!git checkout -- flash_head/utils/cpu_face_handler.py 2>/dev/null

# ═══════════════════════════════════════════════════
# PART 0: Replace CPUFaceHandler with OpenCV (mediapipe is broken)
# ═══════════════════════════════════════════════════
handler_path = "/content/SoulX-FlashHead/flash_head/utils/cpu_face_handler.py"
handler_code = '''import cv2
import numpy as np
from typing import Tuple, List

class CPUFaceHandler:
    """OpenCV-based face detection (replaces broken mediapipe)."""

    def __init__(self, model_selection: int = 1, min_detection_confidence: float = 0.0):
        # Try DNN first (more accurate), fall back to Haar
        proto = "/content/SoulX-FlashHead/deploy.prototxt"
        caffe = "/content/SoulX-FlashHead/res10_300x300_ssd_iter_140000.caffemodel"
        self.use_dnn = False
        self.use_haar = False

        # Download DNN model if not present
        import urllib.request
        if not os.path.exists(proto):
            try:
                urllib.request.urlretrieve(
                    "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt",
                    proto)
            except: pass
        if not os.path.exists(caffe):
            try:
                urllib.request.urlretrieve(
                    "https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel",
                    caffe)
            except: pass

        if os.path.exists(proto) and os.path.exists(caffe):
            self.net = cv2.dnn.readNetFromCaffe(proto, caffe)
            self.use_dnn = True
        else:
            cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
            self.cascade = cv2.CascadeClassifier(cascade_path)
            self.use_haar = True

    def detect(self, image: np.ndarray) -> Tuple[List, List]:
        bboxs, scores = [], []
        img_h, img_w = image.shape[:2]

        if self.use_dnn:
            blob = cv2.dnn.blobFromImage(image, 1.0, (300, 300), (104.0, 177.0, 123.0))
            self.net.setInput(blob)
            detections = self.net.forward()
            for i in range(detections.shape[2]):
                confidence = float(detections[0, 0, i, 2])
                if confidence > 0.5:
                    # Return RELATIVE coordinates (same as original mediapipe handler)
                    x1 = detections[0, 0, i, 3]
                    y1 = detections[0, 0, i, 4]
                    x2 = detections[0, 0, i, 5]
                    y2 = detections[0, 0, i, 6]
                    bboxs.append([float(x1), float(y1), float(x2), float(y2)])
                    scores.append(confidence)
        elif self.use_haar:
            gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
            faces = self.cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))
            for (x, y, w, h) in faces:
                # Convert to RELATIVE coordinates
                bboxs.append([x/img_w, y/img_h, (x+w)/img_w, (y+h)/img_h])
                scores.append(1.0)

        return bboxs, scores

    def __call__(self, image: np.ndarray) -> Tuple[List, List]:
        return self.detect(image)
'''

import os as _os
with open(handler_path, "w") as f:
    f.write("import os\n" + handler_code)
print("✅ Replaced CPUFaceHandler with OpenCV (no mediapipe)")

# ═══════════════════════════════════════════════════
# PART 1: Patch facecrop.py to SAVE the crop bbox
# ═══════════════════════════════════════════════════
facecrop_path = "/content/SoulX-FlashHead/flash_head/utils/facecrop.py"
with open(facecrop_path, "r") as f:
    fc_code = f.read()

fc_code = fc_code.replace(
    'import os',
    'import os\nimport json\n\n_LAST_CROP_BBOX = None'
)

fc_code = fc_code.replace(
    '    crop_face = face_image.crop(scaled_bbox)',
    '    global _LAST_CROP_BBOX\n'
    '    _LAST_CROP_BBOX = scaled_bbox + [img_w, img_h]\n'
    '    try:\n'
    '        with open("/tmp/_flashhead_crop_bbox.json", "w") as _bf:\n'
    '            json.dump({"bbox": scaled_bbox, "img_w": img_w, "img_h": img_h}, _bf)\n'
    '    except: pass\n'
    '    crop_face = face_image.crop(scaled_bbox)'
)

with open(facecrop_path, "w") as f:
    f.write(fc_code)
print("✅ Patched facecrop.py to save crop bbox")

# ═══════════════════════════════════════════════════
# PART 2: Patch gradio_app.py
# ═══════════════════════════════════════════════════
filepath = "/content/SoulX-FlashHead/gradio_app.py"
with open(filepath, "r") as f:
    code = f.read()

# Fix model paths
code = code.replace('value="models/SoulX-FlashHead-1_3B"', 'value="/content/models/SoulX-FlashHead-1_3B"')
code = code.replace('value="models/wav2vec2-base-960h"', 'value="/content/models/wav2vec2-base-960h"')

# Default to lite
code = code.replace('value="pro",', 'value="lite",')

# Default face crop ON
code = code.replace(
    'label="Use Face Crop",\n                value=False,',
    'label="Use Face Crop",\n                value=True,'
)

# Share mode
if 'app.launch(share=True' not in code:
    code = code.replace('app.launch()', 'app.launch(share=True, debug=True)')

# Branding
match = re.search(r'^( *)gr\.Markdown\("# ⚡ SoulX-FlashHead Video Generator"\)', code, re.MULTILINE)
if match:
    old_line = match.group(0)
    ind = match.group(1)
    brand_html = (
        '<div style="text-align:center; padding:25px; '
        'background:linear-gradient(135deg,#667eea 0%,#764ba2 100%); '
        'border-radius:12px; box-shadow:0 8px 16px rgba(0,0,0,0.2); margin-bottom:20px;">'
        '<h1 style="font-size:2.2em; margin:0; color:white;">🎬 SoulX FlashHead Lite - AI Talking Head</h1>'
        '<p style="font-size:1.1em; color:rgba(255,255,255,0.9); margin:6px 0 12px;">'
        'Audio-Driven Portrait Animation | Lite &amp; Pro Models</p>'
        '<div style="display:flex; justify-content:center; gap:12px; flex-wrap:wrap;">'
        '<a href="https://www.youtube.com/@AiQuestacademy" target="_blank" '
        'style="background:#FF0000; color:#fff; padding:8px 18px; border-radius:8px; '
        'text-decoration:none; font-weight:600; font-size:0.95em;">'
        '▶ YouTube - AIQUEST</a>'
        '<a href="https://x.com/AiQuestacademy" target="_blank" '
        'style="background:#000; color:#fff; padding:8px 18px; border-radius:8px; '
        'text-decoration:none; font-weight:600; font-size:0.95em;">'
        '𝕏 Follow on X</a></div>'
        '<p style="margin-top:10px; font-size:0.85em; color:rgba(255,255,255,0.7);">'
        'Notebook by <b>AIQUEST</b> — Subscribe for more AI tools &amp; tutorials!</p></div>'
    )
    new_line = ind + 'gr.HTML(' + repr(brand_html) + ')'
    code = code.replace(old_line, new_line)
    print("✅ Branding injected")

# ── Inject post-processing into run_inference ──
run_inf_pattern = re.search(
    r'(logger\.info\(f"Saved to \{final_video_path\}"\))\s*\n(\s*)(return final_video_path)',
    code
)
if run_inf_pattern:
    old_block = run_inf_pattern.group(0)
    ind = run_inf_pattern.group(2)
    new_block = (
        run_inf_pattern.group(1) + '\n'
        + ind + '# ── 16:9 Aspect Ratio Post-Processing ──\n'
        + ind + 'try:\n'
        + ind + '    final_video_path = _composite_to_original(final_video_path, cond_image)\n'
        + ind + 'except Exception as e:\n'
        + ind + '    import traceback; logger.error(f"COMPOSITE FAILED: {e}\\n{traceback.format_exc()}")\n'
        + ind + 'return final_video_path'
    )
    code = code.replace(old_block, new_block)
    print("✅ Post-processing injected into run_inference")

# ── Inject into run_multi_gpu_inference ──
multi_pattern = re.search(
    r'(if os\.path\.exists\(save_path\):)\s*\n(\s*)(return save_path)',
    code
)
if multi_pattern:
    old_block2 = multi_pattern.group(0)
    ind2 = multi_pattern.group(2)
    new_block2 = (
        multi_pattern.group(1) + '\n'
        + ind2 + '# ── 16:9 Aspect Ratio Post-Processing ──\n'
        + ind2 + 'try:\n'
        + ind2 + '    save_path = _composite_to_original(save_path, cond_image)\n'
        + ind2 + 'except Exception as e:\n'
        + ind2 + '    import traceback; logger.error(f"COMPOSITE FAILED: {e}\\n{traceback.format_exc()}")\n'
        + ind2 + 'return save_path'
    )
    code = code.replace(old_block2, new_block2)
    print("✅ Post-processing injected into run_multi_gpu_inference")

# ── Add composite function ──
first_def = code.index("\ndef ")
composite_func = '''
def _composite_to_original(video_path, original_image_path):
    """
    Reads the EXACT crop bbox saved by facecrop.py during model inference,
    then pastes each 512x512 generated frame back into that exact region.
    """
    import cv2, json, numpy as np, subprocess

    bbox_file = "/tmp/_flashhead_crop_bbox.json"
    if not os.path.exists(bbox_file):
        logger.info("No crop bbox file found — face_crop was likely disabled, skipping composite")
        return video_path

    with open(bbox_file, "r") as f:
        crop_data = json.load(f)

    crop_x1, crop_y1, crop_x2, crop_y2 = crop_data["bbox"]
    orig_img_w = crop_data["img_w"]
    orig_img_h = crop_data["img_h"]

    os.remove(bbox_file)

    aspect = max(orig_img_w, orig_img_h) / max(min(orig_img_w, orig_img_h), 1)
    if aspect < 1.15:
        logger.info(f"Original {orig_img_w}x{orig_img_h} is ~square, skipping composite")
        return video_path

    crop_w = crop_x2 - crop_x1
    crop_h = crop_y2 - crop_y1

    logger.info(f"COMPOSITE: using EXACT model bbox ({crop_x1},{crop_y1})-({crop_x2},{crop_y2}) = {crop_w}x{crop_h} in {orig_img_w}x{orig_img_h}")

    orig = cv2.imread(original_image_path)
    if orig is None:
        logger.warning(f"Could not read original image: {original_image_path}")
        return video_path

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    output_path = video_path.replace(".mp4", "_ar.mp4")

    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (orig_img_w, orig_img_h))

    mask = np.ones((crop_h, crop_w), dtype=np.float32)
    feather = max(min(crop_w, crop_h) // 12, 6)
    for i in range(feather):
        a = i / feather
        mask[i, :] *= a
        mask[-(i+1), :] *= a
        mask[:, i] *= a
        mask[:, -(i+1)] *= a
    mask_3ch = mask[:, :, np.newaxis]

    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        resized = cv2.resize(frame, (crop_w, crop_h), interpolation=cv2.INTER_LANCZOS4)
        canvas = orig.copy()
        roi = canvas[crop_y1:crop_y2, crop_x1:crop_x2].astype(np.float32)
        blended = (resized.astype(np.float32) * mask_3ch + roi * (1 - mask_3ch)).astype(np.uint8)
        canvas[crop_y1:crop_y2, crop_x1:crop_x2] = blended
        out.write(canvas)
        frame_count += 1

    cap.release()
    out.release()

    final_path = output_path.replace(".mp4", "_final.mp4")
    subprocess.run(["ffmpeg", "-y", "-i", output_path, "-i", video_path,
                    "-c:v", "libx264", "-c:a", "aac", "-map", "0:v", "-map", "1:a",
                    "-shortest", final_path], capture_output=True)

    if os.path.exists(final_path) and os.path.getsize(final_path) > 0:
        os.replace(final_path, video_path)
        if os.path.exists(output_path):
            os.remove(output_path)
        logger.info(f"✅ Composited {frame_count} frames -> {orig_img_w}x{orig_img_h}, saved to {video_path}")
        return video_path

    if os.path.exists(output_path):
        os.replace(output_path, video_path)
    return video_path

'''
code = code[:first_def] + composite_func + code[first_def:]

with open(filepath, "w") as f:
    f.write(code)

# ═══════════════════════════════════════════════════
# Verify everything
# ═══════════════════════════════════════════════════
with open(filepath, "r") as f:
    final = f.read()
with open(facecrop_path, "r") as f:
    fc_final = f.read()
with open(handler_path, "r") as f:
    handler_final = f.read()

checks = [
    ("_composite_to_original(final_video_path, cond_image)" in final, "Post-process in run_inference"),
    ("_composite_to_original(save_path, cond_image)" in final, "Post-process in run_multi_gpu"),
    ("def _composite_to_original" in final, "Composite function"),
    ("_flashhead_crop_bbox.json" in final, "Reads exact bbox from model"),
    ("_flashhead_crop_bbox.json" in fc_final, "facecrop.py saves bbox"),
    ("mediapipe" not in handler_final, "CPUFaceHandler uses OpenCV (no mediapipe)"),
    ("cv2.dnn" in handler_final, "DNN face detector in handler"),
    ("value=True," in final and "Use Face Crop" in final, "Face crop defaults ON"),
    ("/content/models/" in final, "Model paths"),
    ("share=True" in final, "Share mode"),
]
print("\n── Verification ──")
for ok, label in checks:
    print(f"{'✅' if ok else '❌'} {label}")

# Test that the handler actually works
print("\n── Handler Test ──")
try:
    import sys
    for mod in list(sys.modules.keys()):
        if "face_handler" in mod or "facecrop" in mod:
            del sys.modules[mod]
    from flash_head.utils.cpu_face_handler import CPUFaceHandler
    handler = CPUFaceHandler()
    print(f"✅ CPUFaceHandler initialized (dnn={handler.use_dnn}, haar={handler.use_haar})")
except Exception as e:
    print(f"❌ CPUFaceHandler FAILED: {e}")

In [ ]:
# @title Quick Fix: Force face_crop default ON
filepath = "/content/SoulX-FlashHead/gradio_app.py"
with open(filepath, "r") as f:
    code = f.read()

import re
# Match the checkbox regardless of whitespace
code = re.sub(
    r'(use_face_crop_input\s*=\s*gr\.Checkbox\([^)]*?)value\s*=\s*False',
    r'\1value=True',
    code
)

with open(filepath, "w") as f:
    f.write(code)

# Verify
with open(filepath, "r") as f:
    final = f.read()

match = re.search(r'use_face_crop_input.*?value\s*=\s*(True|False)', final, re.DOTALL)
print(f"✅ Face crop default: {match.group(1)}" if match else "❌ Not found")

In [ ]:
# @title Step 6: 🚀 Launch SoulX FlashHead Gradio UI
%cd /content/SoulX-FlashHead

print("🚀 Launching SoulX FlashHead Lite...")
print("🔗 Watch for the public Gradio URL below!\n")

!python gradio_app.py

---

<div align="center">

### 🎉 Enjoyed this notebook?

If this was helpful, please **⭐ star the repo** and **subscribe** for more free AI tools & tutorials!

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/▶%20Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20𝕏-0000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

**Made with ❤️ by AIQUEST** | [@aiquestacademy](https://youtube.com/@aiquestacademy)

</div>